# Inter-Annotator Agreement (IAA) — Krippendorff's α via BERTScore

Pipeline:
1. Pull `human_insights` rows from Supabase
2. Build (dashboard_id, chart_id, level) units
3. Compute BERTScore-based pairwise distances for all 3 annotator pairs
4. Compute Krippendorff's α overall, per level, and per dashboard
5. Export results to Excel

## Dependencies

In [ ]:
# ============================================================
# Run in Google Colab with GPU runtime for speed
# ============================================================
!pip install -q bert-score supabase pandas numpy openpyxl

## STEP 1: Pull data from Supabase

In [ ]:
import pandas as pd
import numpy as np
from bert_score import score as bert_score
from supabase import create_client
from itertools import combinations
from google.colab import userdata

# Credentials stored as Colab secrets (same pattern as other notebooks)
SUPABASE_URL = userdata.get("SUPABASE_URL")
SUPABASE_KEY = userdata.get("SUPABASE_KEY")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
response = supabase.table("human_insights").select("*").execute()
df = pd.DataFrame(response.data)

# Inspect columns and shape before proceeding
print("Columns:", df.columns.tolist())
print("Shape  :", df.shape)
df.head(3)

## STEP 2: Build unit identifier

Each **unit** is one `(dashboard_id, chart_id, level)` combination.  
Adjust `ANNOTATOR_COLS` if your column names differ.

In [ ]:
# ── Adjust these to your actual column names if needed ──────
# Expected: dashboard_id, chart_id, level,
#           insight_part_1, insight_part_2, insight_part_3
ANNOTATOR_COLS = ["insight_part_1", "insight_part_2", "insight_part_3"]

df["unit_id"] = (
    df["dashboard_id"].astype(str) + "__"
    + df["chart_id"].astype(str) + "__"
    + df["level"].astype(str)
)

ANNOTATOR_PAIRS = list(combinations(ANNOTATOR_COLS, 2))
# -> [('insight_part_1','insight_part_2'),
#     ('insight_part_1','insight_part_3'),
#     ('insight_part_2','insight_part_3')]

print(f"Annotator pairs  : {ANNOTATOR_PAIRS}")
print(f"Unique units     : {df['unit_id'].nunique()}")
print(f"Unique levels    : {sorted(df['level'].unique())}")
print(f"Unique dashboards: {sorted(df['dashboard_id'].unique())}")

## STEP 3: BERTScore-based distance function

`distance(a, b) = 1 - BERTScore_F1(a, b)`  
`NOT APPLICABLE` entries are treated as missing (`np.nan`).

In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

NA_MARKERS = {"NOT APPLICABLE", "not applicable", "", None}

def is_missing(text):
    return text is None or str(text).strip() in NA_MARKERS


def bertscore_distance_batch(refs, hyps, model_type="distilbert-base-uncased", device=DEVICE):
    """
    Compute BERTScore F1 for parallel lists of refs and hyps.
    Returns a numpy array of distances (1 - F1).
    Missing values return np.nan.
    """
    assert len(refs) == len(hyps)
    distances = np.full(len(refs), np.nan)

    valid_indices = [
        i for i, (r, h) in enumerate(zip(refs, hyps))
        if not is_missing(r) and not is_missing(h)
    ]

    if not valid_indices:
        return distances

    valid_refs = [str(refs[i]) for i in valid_indices]
    valid_hyps = [str(hyps[i]) for i in valid_indices]

    _, _, F1 = bert_score(
        valid_hyps, valid_refs,
        model_type=model_type,
        device=device,
        verbose=False
    )

    for idx, f1_val in zip(valid_indices, F1.numpy()):
        distances[idx] = 1.0 - f1_val

    return distances

## STEP 4: Precompute pairwise BERTScore distances for all units

Adds columns `dist_12`, `dist_13`, `dist_23` to `df`.

In [ ]:
for (col_a, col_b) in ANNOTATOR_PAIRS:
    # e.g. insight_part_1 + insight_part_2 -> dist_12
    suffix_a = col_a.split("_")[-1]   # '1'
    suffix_b = col_b.split("_")[-1]   # '2'
    pair_key = f"dist_{suffix_a}{suffix_b}"
    print(f"Computing {pair_key} ({col_a} vs {col_b}) ...")
    df[pair_key] = bertscore_distance_batch(
        df[col_a].tolist(),
        df[col_b].tolist(),
        device=DEVICE
    )

DIST_COLS = ["dist_12", "dist_13", "dist_23"]
print("\nSample distances:")
print(df[["unit_id"] + DIST_COLS].head(10).to_string(index=False))

## STEP 5: Krippendorff's alpha with custom BERTScore distance

$$\alpha = 1 - \frac{D_o}{D_e}$$

- **D_o** = mean observed disagreement (average pairwise distance within units, >=2 valid annotators)
- **D_e** = mean expected disagreement (average over all valid distances in the pool)

In [ ]:
def krippendorff_alpha_bertscore(sub_df):
    """
    Compute Krippendorff's alpha using BERTScore distances.
    sub_df must contain columns: dist_12, dist_13, dist_23.
    Returns alpha (float) or np.nan if insufficient data.
    """
    # --- Observed disagreement (D_o) ---
    # For each unit: average the available pairwise distances
    unit_disagreements = sub_df[DIST_COLS].mean(axis=1, skipna=True)
    # Only include units where at least one pair has a valid distance
    valid_units = unit_disagreements.notna()
    if valid_units.sum() < 2:
        return np.nan
    D_o = unit_disagreements[valid_units].mean()

    # --- Expected disagreement (D_e) ---
    # Pool all valid pairwise distances and take the mean
    all_distances = sub_df[DIST_COLS].values.flatten()
    all_distances = all_distances[~np.isnan(all_distances)]
    if len(all_distances) == 0:
        return np.nan
    D_e = all_distances.mean()

    if D_e == 0:
        return np.nan  # degenerate: all annotations identical

    return 1.0 - (D_o / D_e)

## STEP 6: Overall alpha (all dashboards combined)

In [ ]:
alpha_overall = krippendorff_alpha_bertscore(df)
print(f"Krippendorff's alpha (BERTScore, overall): {alpha_overall:.4f}")

## STEP 7: Alpha per insight level (L2 / L3 / L4)

In [ ]:
results_level = []
for level in ["L2", "L3", "L4"]:
    sub = df[df["level"] == level]
    alpha = krippendorff_alpha_bertscore(sub)
    n_units = sub.shape[0]
    n_valid = sub[DIST_COLS].notna().any(axis=1).sum()
    results_level.append({
        "Level"        : level,
        "Alpha"        : round(alpha, 4) if not np.isnan(alpha) else "N/A",
        "N_units_total": n_units,
        "N_units_valid": n_valid,
    })

level_df = pd.DataFrame(results_level)
print("--- Alpha by Level ---")
print(level_df.to_string(index=False))

## STEP 8: Alpha per dashboard

In [ ]:
results_dash = []
for dash_id in sorted(df["dashboard_id"].unique()):
    sub = df[df["dashboard_id"] == dash_id]
    alpha = krippendorff_alpha_bertscore(sub)
    results_dash.append({
        "Dashboard": dash_id,
        "Alpha"    : round(alpha, 4) if not np.isnan(alpha) else "N/A",
        "N_units"  : sub.shape[0],
    })

dash_df = pd.DataFrame(results_dash)
print("--- Alpha by Dashboard ---")
print(dash_df.to_string(index=False))

## STEP 9: Alpha per dashboard x level

In [ ]:
results_cross = []
for dash_id in sorted(df["dashboard_id"].unique()):
    for level in ["L2", "L3", "L4"]:
        sub = df[(df["dashboard_id"] == dash_id) & (df["level"] == level)]
        alpha = krippendorff_alpha_bertscore(sub)
        results_cross.append({
            "Dashboard": dash_id,
            "Level"    : level,
            "Alpha"    : round(alpha, 4) if not np.isnan(alpha) else "N/A",
            "N_units"  : sub.shape[0],
        })

cross_df = pd.DataFrame(results_cross)
print("--- Alpha by Dashboard x Level ---")
pivot = cross_df.pivot(index="Dashboard", columns="Level", values="Alpha")
print(pivot.to_string())

## STEP 10: Summary printout

In [ ]:
print("=" * 50)
print("INTER-ANNOTATOR AGREEMENT SUMMARY")
print("=" * 50)
print(f"\nOverall Krippendorff's alpha : {alpha_overall:.4f}")

interp = (
    "poor (< 0.20)"            if alpha_overall < 0.20 else
    "fair (0.20-0.40)"         if alpha_overall < 0.40 else
    "moderate (0.40-0.60)"     if alpha_overall < 0.60 else
    "substantial (0.60-0.80)"  if alpha_overall < 0.80 else
    "near-perfect (>= 0.80)"
)
print(f"Interpretation              : {interp}")

print("\n--- By Level ---")
print(level_df.to_string(index=False))

print("\n--- By Dashboard ---")
print(dash_df.to_string(index=False))

## STEP 11: Export results to Excel

In [ ]:
OUTPUT_FILE = "iaa_krippendorff_bertscore.xlsx"

# Add overall row to level summary sheet
level_df_export = pd.concat([
    level_df,
    pd.DataFrame([{
        "Level"        : "OVERALL",
        "Alpha"        : round(alpha_overall, 4) if not np.isnan(alpha_overall) else "N/A",
        "N_units_total": df.shape[0],
        "N_units_valid": df[DIST_COLS].notna().any(axis=1).sum(),
    }])
], ignore_index=True)

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
    level_df_export.to_excel(writer, sheet_name="By_Level",            index=False)
    dash_df.to_excel(        writer, sheet_name="By_Dashboard",         index=False)
    cross_df.to_excel(       writer, sheet_name="By_Dashboard_x_Level", index=False)
    df.to_excel(             writer, sheet_name="Raw_Distances",         index=False)

print(f"Saved: {OUTPUT_FILE}")
print("Sheets: By_Level | By_Dashboard | By_Dashboard_x_Level | Raw_Distances")